# Diabetic Retinopathy Detection

In [11]:
import cv2,os
data_path='dataset/'
categories=os.listdir(data_path)

labels=[i for i in range(len(categories))]

label_dict=dict(zip(categories,labels)) #empty dictionary
print(label_dict)
print(categories)
print(labels)

{'cataract': 0, 'diabetic_retinopathy': 1, 'disc_edema': 2, 'glaucoma': 3, 'normal': 4, 'retinal_detachment': 5}
['cataract', 'diabetic_retinopathy', 'disc_edema', 'glaucoma', 'normal', 'retinal_detachment']
[0, 1, 2, 3, 4, 5]


In [21]:
!pip install opencv-python


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
img_size_x=128
img_size_y=128
data=[]
label=[]

for category in categories:
    folder_path=os.path.join(data_path,category)
    img_names=os.listdir(folder_path)
        
    for img_name in img_names:
        img_path=os.path.join(folder_path,img_name)
        img=cv2.imread(img_path)
        try:
            img = cv2.imread(img_path)
            img = cv2.resize(img,(img_size_x,img_size_y))
            data.append(img)
            label.append(label_dict[category])
            #appending the image and the label(categorized) into the list (dataset)
        except Exception as e:
            print('Exception:',e)
            #if any exception rasied, the exception will be printed here. And pass to the next image

Exception: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'

Exception: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'

Exception: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'



# Recale and assign  catagorical labels

In [24]:
import numpy as np

data = np.array(data) / 255.0
data = np.reshape(data,(data.shape[0],img_size_x,img_size_y,3))

label = np.array(label)

from tensorflow.keras.utils import to_categorical
new_label = to_categorical(label)

In [25]:
!pip install tensorflow


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# CNN Model

In [26]:
data.shape

(3179, 128, 128, 3)

In [27]:
data.shape[1:]

(128, 128, 3)

In [42]:
from keras.models import Sequential
from keras.layers import Dense,Activation,Flatten,Dropout
from keras.layers import Conv2D,MaxPooling2D
import tensorflow as tf
from tensorflow.keras import layers

model = tf.keras.Sequential([
    layers.Conv2D(8, (3,3), padding="valid", input_shape=(128,128,3), activation = 'relu'),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.BatchNormalization(),
    
    layers.Conv2D(16, (3,3), padding="valid", activation = 'relu'),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.BatchNormalization(),
    
    layers.Conv2D(32, (4,4), padding="valid", activation = 'relu'),
    layers.MaxPooling2D(pool_size=(2,2)),
    layers.BatchNormalization(),
 
    layers.Flatten(),
    layers.Dense(64, activation = 'relu'),
    layers.Dropout(0.15),
    layers.Dense(6, activation = 'softmax')
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate = 1e-4), loss=tf.keras.losses.categorical_crossentropy, metrics=['acc'])

C:\Users\nitya\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [31]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)                    │ (None, 126, 126, 8)         │             224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_9 (MaxPooling2D)       │ (None, 63, 63, 8)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_9                │ (None, 63, 63, 8)           │              32 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_10 (Conv2D)                   │ (None, 61, 61, 16)          │           1,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_10 (MaxPooling2D)      │ (None, 30, 30, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_10               │ (None, 30, 30, 16)          │              64 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_11 (Conv2D)                   │ (None, 27, 27, 32)          │           8,224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_11 (MaxPooling2D)      │ (None, 13, 13, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_11               │ (None, 13, 13, 32)          │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_3 (Flatten)                  │ (None, 5408)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 64)                  │         346,176 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_7 (Dense)                      │ (None, 6)                   │             390 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 356,406 (1.36 MB)

 Trainable params: 356,294 (1.36 MB)

 Non-trainable params: 112 (448.00 B)

# Splitting data into traning and testing

In [39]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(data,new_label,test_size=0.2)

In [34]:
!pip install scikit-learn


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [41]:
history = model.fit(x_train, y_train, epochs=10, batch_size=16, validation_data=(x_test, y_test))

Epoch 1/10


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 5), output.shape=(None, 6)

In [ ]:
cnn_acc = max(history.history['val_acc'])
print("CNN Accuracy:", cnn_acc)

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score


# Model evaluation
y_pred = model.predict(x_test) 

# Convert one-hot encoded labels back to categorical labels
y_true_labels = np.argmax(y_test, axis=1)
y_pred_labels = np.argmax(y_pred, axis=1)

# Confusion matrix
conf_mat = confusion_matrix(y_true_labels, y_pred_labels)
print("Confusion Matrix:")
print(conf_mat)

# Specificity (True Negative Rate), Precision, Recall, and F1 Score for each class
for i in range(conf_mat.shape[0]):
    true_negative = np.sum(conf_mat) - np.sum(conf_mat[i, :]) - np.sum(conf_mat[:, i]) + conf_mat[i, i]
    false_positive = np.sum(conf_mat[:, i]) - conf_mat[i, i]
    false_negative = np.sum(conf_mat[i, :]) - conf_mat[i, i]
    true_positive = conf_mat[i, i]

    specificity = true_negative / (true_negative + false_positive)
    
    # Check for zero denominators in precision and recall
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    
    # Check for zero denominators in F1 score
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"\nMetrics for Class {i}:")
    print("Specificity (True Negative Rate): {:.4f}".format(specificity))
    print("Precision: {:.4f}".format(precision))
    print("Recall: {:.4f}".format(recall))
    print("F1 Score: {:.4f}".format(f1))



In [ ]:
import matplotlib.pyplot as plt

# # plot the training loss and accuracy
N = 10 #number of epochs
plt.style.use("ggplot")
plt.figure()
plt.plot(np.arange(0, N), history.history["loss"], label="train_loss")
plt.plot(np.arange(0, N), history.history["val_loss"], label="val_loss")
plt.plot(np.arange(0, N), history.history["acc"], label="train_acc")
plt.plot(np.arange(0, N), history.history["val_acc"], label="val_acc")
plt.title("Training Loss and Accuracy")
plt.xlabel("Epoch #")
plt.ylabel("Loss/Accuracy")
plt.legend(loc="center right")

In [ ]:
!pip install matplotlib

In [ ]:
model.save('model.h5')

In [ ]:
plt.figure()
plt.plot(history.history['acc'], label='Train Accuracy')
plt.plot(history.history['val_acc'], label='Validation Accuracy')
plt.legend()
plt.title('Accuracy Graph')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.show()

In [ ]:
plt.figure()
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.legend()
plt.title('Loss Graph')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

In [ ]:
import seaborn as sns

plt.figure(figsize=(6,5))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Blues',
            xticklabels=categories,
            yticklabels=categories)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
!pip install seaborn

In [ ]:
precision_list = []
recall_list = []
f1_list = []

for i in range(conf_mat.shape[0]):
    tp = conf_mat[i,i]
    fp = np.sum(conf_mat[:,i]) - tp
    fn = np.sum(conf_mat[i,:]) - tp

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)

    precision_list.append(precision)
    recall_list.append(recall)
    f1_list.append(f1)

# x-axis positions
x = np.arange(len(categories))

# 🔥 improved figure
plt.figure(figsize=(10,4))

plt.bar(x - 0.2, precision_list, width=0.2, label='Precision')
plt.bar(x, recall_list, width=0.2, label='Recall')
plt.bar(x + 0.2, f1_list, width=0.2, label='F1 Score')

# 🔥 FIXED LABELS
plt.xticks(x, categories, rotation=45, ha='right')

plt.xlabel("Classes")
plt.ylabel("Score")
plt.title("Precision, Recall, F1 Score")

plt.legend()

# 🔥 spacing fix
plt.tight_layout()

plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [ ]:
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2   # 80 train, 20 test
)

In [ ]:
train_generator = datagen.flow_from_directory(
    'dataset',   # tumhara main folder
    target_size=(224,224),
    batch_size=16,
    class_mode='categorical',
    subset='training'
)

In [ ]:
test_generator = datagen.flow_from_directory(
    'dataset',
    target_size=(224,224),
    batch_size=16,
    class_mode='categorical',
    subset='validation'
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout

# Base model
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze some layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

for layer in base_model.layers[-20:]:
    layer.trainable = True

# Add custom layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(6, activation='softmax')(x)

# Final model
vit_model = Model(inputs=base_model.input, outputs=output)

# Compile
vit_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history_vit = vit_model.fit(
    train_generator,
    epochs=10,
    validation_data=test_generator
)

In [ ]:
vit_acc = max(history_vit.history['val_accuracy'])
print("ViT Accuracy:", vit_acc)

In [ ]:
vit_model.save("mobilenet_model.h5")

In [ ]:
model.save("cnn_model.h5")

In [ ]:
cnn_acc = max(history.history['val_acc'])
vit_acc = max(history_vit.history['val_accuracy'])

print("CNN Accuracy:", cnn_acc)
print("MobileNet Accuracy:", vit_acc)

In [ ]:
import matplotlib.pyplot as plt

models = ['CNN', 'MobileNet']
accuracies = [cnn_acc, vit_acc]

plt.figure()
plt.bar(models, accuracies)
plt.title("CNN vs MobileNet Accuracy")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
# sample image
sample = x_test[0]

# CNN input
cnn_input = np.reshape(sample, (1,128,128,3))

# MobileNet input
import cv2
sample_m = cv2.resize(sample, (224,224))
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
sample_m = preprocess_input(sample_m)
sample_m = np.reshape(sample_m, (1,224,224,3))

# predictions
cnn_pred = model.predict(cnn_input)
vit_pred = vit_model.predict(sample_m)

# fusion 
cnn_conf = np.max(cnn_pred)
vit_conf = np.max(vit_pred)

cnn_w = cnn_conf / (cnn_conf + vit_conf)
vit_w = vit_conf / (cnn_conf + vit_conf)

final_pred = (cnn_w * cnn_pred) + (vit_w * vit_pred)

print("CNN:", np.argmax(cnn_pred))
print("MobileNet:", np.argmax(vit_pred))
print("FINAL:", np.argmax(final_pred))